# Choropleth Analysis: Categorical Correlation

Purpose: The purpose of this notebook is to examine the geographic relationship between net migration rates and county health scores using univariate and bivariate choropleth maps\.

Inputs
1\. Cleaned county\-to\-county migration data: 'county\_to\_county\_US\_2020\.csv'
2\. Cleaned health rankings data: 'cleaned\_health\_rankings\_2025\.csv'

Outputs
None

How to Run
Run after notebooks 1\. Setup: Importing Dependencies and 2\. Ingesting and Cleaning Data
Optionally run before notebook 4\. Overall Health\-Migration Relationship

In [1]:
import os

import requests
import pandas as pd
import numpy as np
import plotly.express as px

## Visualizing Net Migration

### Read in migration data and create county\-level summaries\. 

In [2]:
county_to_county_migration_df = pd.read_csv('data/county_to_county_US_2020.csv', dtype={
    'GEOID1': str,
    'GEOID2': str,
    'state': str,
    'county': str
    })

See functions\.py for create\_county\_summaries codebase\. The function uses groupby\(\)\.agg\(\) to create a dataframe with one row per county that sums MOVEDIN, sums MOVEDOUT, and sums MOVEDNET\. It also calculates in\-, out\-, and net\-migration rates per 1000 residents\.  

In [3]:
from functions import create_county_summaries

county_summaries_df = create_county_summaries()

county_summaries_df.head(3)

,GEOID1,MOVEDIN,MOVEDOUT,MOVEDNET,POP1YR,POP1YRAGO,STATE1_NAME,FULL1_NAME,Net_Migration_Rate,In_Migration_Rate,Out_Migration_Rate
0,01001,3375.0,6501.0,-3126.0,54929.0,57775.0,Alabama,"Autauga County, Alabama",-56.909829,61.442954,118.352783
1,01003,10770.0,8779.0,1991.0,216518.0,213875.0,Alabama,"Baldwin County, Alabama",9.195540,49.741823,40.546283
2,01005,1522.0,1481.0,41.0,24792.0,24683.0,Alabama,"Barbour County, Alabama",1.653759,61.390771,59.737012


### Choropleth of Net Migration Rate Across US Counties

In [4]:
mig_map_df = county_summaries_df.copy()
# Apply signed square root transformation to net migration rate. 
mig_map_df["Net_Power"] = np.sign(mig_map_df["Net_Migration_Rate"]) * np.sqrt(np.abs(mig_map_df["Net_Migration_Rate"]))

Note: Because net migration rates are highly concentrated near zero with a small number of extreme positive or negative values, a linear color scale obscures meaningful geographic variation\. A signed square\-root transformation compresses extreme values and expands variation among typical counties while preserving direction and rank\. This improves the choropleth's effectiveness for a small sacrifice in expressiveness\.

In [5]:
univariate_color_scale = [
        "#67001f",   # deep red
        "#b2182b",
        "#d6604d",
        "#f4a582",
        "#ffffff",   # hard white center
        "#92c5de",
        "#4393c3",
        "#2166ac",
        "#053061"    # deep blue
    ]

def migration_choropleth(color_var='Net_Power'):

    color_scale = univariate_color_scale

    fig = px.choropleth(
        mig_map_df,
        locations="GEOID1",
        geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
        color=color_var,
        scope="usa",
        color_continuous_scale=color_scale,
        color_continuous_midpoint=0,
        hover_name="county_name" if "county_name" in mig_map_df.columns else None,
        hover_data={
            "Net_Migration_Rate": ":.2f",
            "MOVEDNET": ":.0f",
            "MOVEDIN": ":.0f",
            "MOVEDOUT": ":.0f",
        },
    )

    fig.update_geos(visible=False)
    fig.update_layout(
        margin=dict(l=0, r=0, t=40, b=0),
    )

    min_max_color = mig_map_df[color_var].abs().quantile(0.80)
    fig.update_traces(zmin=-min_max_color, zmax=min_max_color,
    marker_line_width=0.1, marker_line_color="rgba(80,80,80,0.3)")

    return fig

In [6]:
net_migration_rate_map = migration_choropleth('Net_Power')
net_migration_rate_map.show()

Analysis: Some geographic patterns appear, such as net out\-migration from sparsely populated Great Plains counties and parts of the post\-industrial Midwest, consistent with long\-running patterns of limited local opportunity\. However, relative to the choropleths below showing population health and community conditions, migration patterns show less geographic structure\. 

## Visualizing County Health and Community Conditions

### Read in county health rankings data\.

In [7]:
health_df = pd.read_csv('data/cleaned_health_rankings_2025.csv', index_col=0, dtype = {'FIPS': str})

### Choropleth of County Population Health Z\-Scores

In [8]:
health_map_df = health_df.copy()

In [9]:
def health_rankings_choropleth(color_var='Health_Z_Score'):

    color_scale = univariate_color_scale

    fig = px.choropleth(
        health_map_df,
        locations="FIPS",
        geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
        color=color_var,
        scope="usa",
        color_continuous_scale=color_scale,
        color_continuous_midpoint=0,
        hover_name="County" if "County" in health_map_df.columns else None,
        hover_data={
            "Health_Z_Score": ":.2f",
            "Community_Z_Score": ":.2f"
        }
    )

    fig.update_geos(visible=False)
    fig.update_layout(
        margin=dict(l=0, r=0, t=40, b=0),
    )

    min_max_color = health_map_df[color_var].abs().quantile(0.90)
    fig.update_traces(zmin=-min_max_color, zmax=min_max_color,
    marker_line_width=0.1, marker_line_color="rgba(80,80,80,0.3)")

    return fig

In [10]:
population_health_map = health_rankings_choropleth('Health_Z_Score')
population_health_map.show()

Analysis: In contrast to intra\-US migration, county health scores show clear geographic structure\. There are broad regions of good health in the upper Midwest, Rockies, and Northeastern and California coasts\. Poor health in the Mississippi Delta is associated with long\-standing economic disadvantage and limited healthcare access\. Additional poor health clusters appear in Central Appalachia and in or around Native American reservations, where geographic isolation and persistent resource constraints contribute to worse population health\. These patterns suggest that community health conditions are more strongly shaped by place\-based structural factors\.

### Choropleth of County Community Conditions Z\-Scores

In [11]:
community_conditions_map = health_rankings_choropleth('Community_Z_Score')
community_conditions_map.show()

Analysis: In contrast to intra\-US migration, county health scores show clear geographic structure\. There are broad regions of good community conditions in the upper Midwest, Rockies, and Northeastern coast\. Poor community conditions in the Mississippi Delta are associated with long\-standing economic disadvantage and limited healthcare access\. Additional poor\-conditions clusters persist in Central Appalachia and in or around Native American reservations, where geographic isolation and persistent resource constraints contribute to worse population health\. One interesting region is California's Central Valley, which has average population health but poor community conditions\. This suggests that the youth of the region's large agricultural labor workforce produces relatively favorable health outcomes despite impoverished and uninsured populations\. Taken together, these patterns suggest that community health conditions are likewise strongly shaped by place\-based structural factors\.

## Visualizing the Relationship Between Migration and Health

In [12]:
from functions import merge_migration_health_ranking_data

merged_mig_health_df = merge_migration_health_ranking_data()

merged_mig_health_df.head(3)

,GEOID1,MOVEDIN,MOVEDOUT,MOVEDNET,POP1YR,POP1YRAGO,STATE1_NAME,FULL1_NAME,Net_Migration_Rate,In_Migration_Rate,...,FIPS,State,County,Number of Counties Included in Health Groups,Health_Z_Score,Health Group,Health Group Range,Community_Z_Score,Health Group.1,Health Group Range.1
0,01001,3375.0,6501.0,-3126.0,54929.0,57775.0,Alabama,"Autauga County, Alabama",-56.909829,61.442954,...,01001,Alabama,Autauga,67,-0.040894,5.0,-0.05 to 0.27,0.114428,5.0,-0.17 to 0.03
1,01003,10770.0,8779.0,1991.0,216518.0,213875.0,Alabama,"Baldwin County, Alabama",9.195540,49.741823,...,01003,Alabama,Baldwin,67,0.309818,4.0,-0.38 to -0.06,0.386812,3.0,-0.58 to -0.37
2,01005,1522.0,1481.0,41.0,24792.0,24683.0,Alabama,"Barbour County, Alabama",1.653759,61.390771,...,01005,Alabama,Barbour,67,-1.116797,8.0,0.96 to 1.38,-0.822974,9.0,0.73 to 1.09


In [13]:
def create_mig_health_terciles(df=merged_mig_health_df, mig_col = 'Net_Migration_Rate'):

    df["mig_bin"] = pd.qcut(df[mig_col], 3, labels=["low mig", "mid mig", "high mig"])
    df["health_bin"] = pd.qcut(df['Health_Z_Score'], 3, labels=["poor health", "mid health", "good health"])
    df["community_bin"] = pd.qcut(df['Community_Z_Score'], 3, labels=["poor cc", "mid cc", "good cc"])

    df["health_bi_class"] = df["health_bin"].astype(str) + " / " + df["mig_bin"].astype(str)
    df["community_bi_class"] = df["community_bin"].astype(str) + " / " + df["mig_bin"].astype(str)

    return df

Note: The choice to divide net migration rates and z\-scores into terciles is again a decision of effectiveness vs\. expressiveness\. Naturally, dividing the data into only three buckets loses important variation\. However, limiting the number of quantiles enabled the choropleths below to visually emphasize those counties where net migration rate and health z\-scores are categorically correlated, albeit with very large categories\. 

In [14]:
mig_health_terciles_df = create_mig_health_terciles(merged_mig_health_df, 'Net_Migration_Rate')

In [15]:
def mig_health_choropleth(color_var='health_bi_class'):

    bivar_colors = {
        # ---- POOR HEALTH ROW (reds) ----
        "poor health / low mig": "#8b0000",  # dark red
        "poor health / mid mig": "#e06666",  # light red
        "poor health / high mig": "#f4a3a3", # very light red

        # ---- MID HEALTH ROW (beiges) ----
        "mid health / low mig":    "#f2e5c4",   # light beige
        "mid health / mid mig":    "#a1926a",   # dark beige
        "mid health / high mig":   "#f2e5c4",   # light beige

        # ---- GOOD HEALTH ROW (greens) ----
        "good health / low mig":   "#deebf7",   # very light blue
        "good health / mid mig":   "#6baed6",   # medium blue
        "good health / high mig":  "#08519c"    # dark blue
    }

    fig = px.choropleth(
        mig_health_terciles_df,
        locations="GEOID1",
        geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
        color=color_var,
        scope="usa",
        color_discrete_map=bivar_colors,
        category_orders={
            'health_bi_class': ['poor health / low mig', 'poor health / mid mig', 'poor health / high mig',
            'mid health / mid mig','mid health / low mig', 'mid health / high mig',
            'good health / high mig', 'good health / mid mig', 'good health / mid low']
            },
        hover_name="County" if "County" in mig_health_terciles_df.columns else None,
        hover_data={
            "health_bi_class": True,
            "GEOID1": False,
            "Net_Migration_Rate": ":.2f",
            "Health_Z_Score": ":.2f",
            "MOVEDIN": True,
            "MOVEDOUT": True
        }
    )

    fig.update_geos(visible=False)
    fig.update_layout(
        title=f"Bivariate Choropleth: Net Migration Rate vs Population Health Score (terciles)",
        margin=dict(l=0, r=0, t=40, b=0)
    )

    fig.update_traces(marker_line_width=0.1, marker_line_color="rgba(0,0,0,0.2)")

    return fig

In [16]:
migration_pop_health_map = mig_health_choropleth('health_bi_class')
migration_pop_health_map.show()

Analysis
Overall Structure: Relative to intra\-US migration, this map shows more geographic structure\. Relative to county health scores, this map shows less geographic structure\. This suggests that while the health\-migration relationship is influenced by place\-based structural factors, that relationship is far less deterministic than community health on its own\. 

Poor Health v\. Good Health Counties: Looking at poor health and good health counties separately, we can see clear regions where poor health is categorically correlated to negative net migration\. In contrast, counties where good health is categorically correlated to positive net migration are both more sparsely populated and less clustered\. This suggests that the relationship between health and migration is notably stronger among poor health counties\. 

Poor Health Centers Remain: Although less regionally consistent, the aforementioned poor\-health centers of the Mississippi Delta, Appalachia, and Native American reservations remain noticeable\. Additionally, we can more clearly see poor health correlated with out migration in the rural Southeast\.

In [17]:
def mig_community_choropleth(color_var='community_bi_class'):

    bivar_colors = {
        # ---- POOR CC ROW (reds) ----
        "poor cc / low mig":   "#8b0000",   # dark red
        "poor cc / mid mig":   "#e06666",   # medium red
        "poor cc / high mig":  "#f4a3a3",   # very light red

        # ---- MID CC (beiges) ----
        "mid cc / low mig":    "#f2e5c4",   # light beige
        "mid cc / mid mig":    "#a1926a",   # dark beige
        "mid cc / high mig":   "#f2e5c4",   # light beige

        # ---- GOOD CC (blues) ----
        "good cc / low mig":   "#deebf7",   # very light blue
        "good cc / mid mig":   "#6baed6",   # medium blue
        "good cc / high mig":  "#08519c"    # dark blue
    }

    fig = px.choropleth(
        mig_health_terciles_df,
        locations="GEOID1",
        geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
        color=color_var,
        scope="usa",
        color_discrete_map=bivar_colors,
        category_orders={
            'community_bi_class': ['poor cc / low mig', 'poor cc / mid mig', 'poor cc / high mig',
            'mid cc / low mig', 'mid cc / mid mig', 'mid cc / high mig',
            'good cc / low mig', 'good cc / mid mig', 'good cc / high mig']
            },
        hover_name="County" if "County" in mig_health_terciles_df.columns else None,
        hover_data={
            "community_bi_class": True,
            "GEOID1": False,
            "Net_Migration_Rate": ":.2f",
            "Community_Z_Score": ":.2f",
            "MOVEDIN": True,
            "MOVEDOUT": True
        }
    )

    fig.update_geos(visible=False)
    fig.update_layout(
        title=f"Bivariate Choropleth: Net Migration Rate vs Community Conditions Score (terciles)",
        margin=dict(l=0, r=0, t=40, b=0),
        legend_title_text="Migration / Community (terciles)"
    )

    fig.update_traces(marker_line_width=0.1, marker_line_color="rgba(0,0,0,0.2)")

    return fig

In [18]:
migration_community_conditions_map = mig_community_choropleth('community_bi_class')
migration_community_conditions_map.show()

Analysis
Overall Structure: Relative to intra\-US migration, this map shows more geographic structure\. Relative to county community conditions scores, this map shows less geographic structure\. This suggests that while the community\-conditions\-to\-migration relationship is influenced by place\-based structural factors, that relationship is far less deterministic than community conditions on its own\. 

Poor Health v\. Good Health Counties: Looking at poor conditions and good conditions counties separately, we can see clear regions where poor conditions are categorically correlated to negative net migration\. In contrast, counties where good conditions are categorically correlated to positive net migration are both more sparsely populated and less clustered\. This suggests that the relationship between community conditions and migration is notably stronger among poor conditions counties\. 

Poor Health Centers Remain: Although less regionally consistent, the aforementioned poor\-conditions centers of the Mississippi Delta, Central Appalachia, and Native American reservations remain noticeable\. Additionally, we can more clearly see poor community conditions correlated with out migration in some parts of the rural Southeast\. 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fba226e9-ff4f-4eec-8dfc-4f877d29c8c6' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>